# Citadel — Anthropic surface validation

Validates the `POST /anthropic/v1/messages` APIM surface end-to-end:

1. JWT auth (D1) — v2.0 issuer, Claude Code audience, `oid` projection.
2. Non-streaming `usage.input_tokens` + `usage.output_tokens` mapping → Event Hub `promptTokens`/`responseTokens`/`totalTokens`.
3. SSE streaming — terminal `message_delta` carries the final `output_tokens`.
4. APIM strips inbound `Authorization` before Foundry call (D1).
5. `userOid`+`userUpn` appear on Event Hub messages.

**This is a stub.** Live execution requires the gates listed below.

## Validation gate — required before running any cell live

| # | Input from the customer | Variable |
|---|------------------------|----------|
| 1 | Entra tenant ID | `CUSTOMER_TENANT_ID` |
| 2 | Claude Code Anthropic-published app ID (audience) | `CLAUDE_CODE_APP_ID` |
| 3 | APIM gateway URL | `APIM_GATEWAY_URL` |
| 4 | Foundry Anthropic endpoint | `FOUNDRY_ANTHROPIC_ENDPOINT` |
| 5 | Foundry Claude deployment name (e.g. `claude-sonnet-4`) | `CLAUDE_DEPLOYMENT_NAME` |
| 6 | Test user JWT (Bearer) | `TEST_USER_JWT` |

Set these as environment variables or in a `.env` next to this notebook before un-skipping the cells.

In [ ]:
# Gate: skip everything if env not configured.
import os, pytest
REQUIRED = ['CUSTOMER_TENANT_ID','CLAUDE_CODE_APP_ID','APIM_GATEWAY_URL','FOUNDRY_ANTHROPIC_ENDPOINT','CLAUDE_DEPLOYMENT_NAME','TEST_USER_JWT']
missing = [k for k in REQUIRED if not os.getenv(k)]
if missing:
    print(f'SKIP — missing env: {missing}')
else:
    print('All env present — ready to execute.')

In [ ]:
# Test 1: non-streaming happy path. Asserts 200 + usage mapping.
import os, requests
if not all(os.getenv(k) for k in REQUIRED):
    raise RuntimeError('gates not satisfied')
r = requests.post(
    f"{os.environ['APIM_GATEWAY_URL']}/anthropic/v1/messages",
    headers={'Authorization': f"Bearer {os.environ['TEST_USER_JWT']}", 'Content-Type': 'application/json'},
    json={'model': os.environ['CLAUDE_DEPLOYMENT_NAME'], 'max_tokens': 64,
          'messages': [{'role':'user','content':'Reply with the single word OK.'}]},
)
assert r.status_code == 200, r.text
body = r.json()
assert 'usage' in body and 'input_tokens' in body['usage'] and 'output_tokens' in body['usage']
print('non-streaming OK', body['usage'])

In [ ]:
# Test 2: streaming — parse SSE, ensure terminal message_delta has output_tokens.
import os, requests, json
r = requests.post(
    f"{os.environ['APIM_GATEWAY_URL']}/anthropic/v1/messages",
    headers={'Authorization': f"Bearer {os.environ['TEST_USER_JWT']}", 'Content-Type': 'application/json'},
    json={'model': os.environ['CLAUDE_DEPLOYMENT_NAME'], 'max_tokens': 64, 'stream': True,
          'messages': [{'role':'user','content':'Count to five.'}]},
    stream=True,
)
assert r.status_code == 200
saw_message_start = False; final_out = None
evt = None
for line in r.iter_lines(decode_unicode=True):
    if not line: continue
    if line.startswith('event:'): evt = line[6:].strip()
    elif line.startswith('data:'):
        d = json.loads(line[5:].strip())
        if evt == 'message_start': saw_message_start = True
        if evt == 'message_delta' and 'usage' in d: final_out = d['usage'].get('output_tokens')
assert saw_message_start and final_out is not None
print('streaming OK, final output_tokens =', final_out)

In [ ]:
# Test 3: bad JWT → 401.
r = requests.post(
    f"{os.environ['APIM_GATEWAY_URL']}/anthropic/v1/messages",
    headers={'Authorization': 'Bearer not-a-real-jwt'},
    json={'model': os.environ['CLAUDE_DEPLOYMENT_NAME'], 'max_tokens': 8,
          'messages': [{'role':'user','content':'hi'}]},
)
assert r.status_code == 401, r.text
print('bad-JWT 401 OK')